# Olist E-Commerce: Cohort & Retention Analysis
**Notebook 1: Data Cleaning & Star Schema**

**Business question:** Are customers acquired in 2017 still buying in 2018?

**Approach:** inspect raw data, clean it, then model a star schema (Customers, Orders, Payments) before any analysis.

## 1. Setup
Define project paths with `pathlib` so the notebook runs on any machine. Raw data is never modified. Generated tables go to `data/processed/`.

In [1]:
import os
from pathlib import Path
import pandas as pd

# Notebook lives in /notebooks, so project root is one level up
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

for d in (PROCESSED_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Files in raw:", sorted(os.listdir(RAW_DIR)))

Project root: c:\Users\HP\Desktop\Toluwalase Aderemi\Career\Programs_or_Jobs\Elevvo\Business_insights_olist
Files in raw: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## 2. Load & Inspect Core Tables
Loading `customers`, `orders` and `payments`. Checking shapes, nulls, order statuses, date range and customer identity.

In [2]:
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
orders    = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
payments  = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")

for name, df in [("customers", customers), ("orders", orders), ("payments", payments)]:
    print(f"--- {name} {df.shape}")
    print(df.isna().sum().loc[lambda s: s > 0], "\n")

print(orders.order_status.value_counts(), "\n")
print("Date range:", orders.order_purchase_timestamp.min(), "->", orders.order_purchase_timestamp.max())
print("customer_id:", customers.customer_id.nunique(), "| customer_unique_id:", customers.customer_unique_id.nunique())

--- customers (99441, 5)
Series([], dtype: int64) 

--- orders (99441, 8)
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64 

--- payments (103886, 5)
Series([], dtype: int64) 

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64 

Date range: 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
customer_id: 99441 | customer_unique_id: 96096


### Findings from initial inspection
- `customer_id` is unique per order. The true customer key is **`customer_unique_id`** (99,441 order-customer IDs vs 96,096 real customers).
- Upper bound on repeat buyers is about 3.5%, so retention will be low across all cohorts.
- `payments` has more rows than `orders`, since an order can have multiple payments. Payments must be aggregated to order level before joining.
- Nulls in approval/delivery dates correspond to non-delivered orders.
- 97% of orders are `delivered`. **Decision:** only delivered orders count as purchases.
- Date range starts Sep 2016 and ends Oct 2018. Edge months are likely too sparse and will be checked before trimming.

## 3. Define Purchases & Choose the Cohort Window
- Keep **delivered** orders only (a purchase = a completed order).
- Attach `customer_unique_id` to each order, since it's the real customer key.
- Check monthly volume to decide which edge months to trim.
- Measure the exact repeat-buyer rate.

In [3]:
delivered = (
    orders[orders.order_status == "delivered"]
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
)
print("Delivered orders:", delivered.shape[0], "| missing unique id:", delivered.customer_unique_id.isna().sum())

delivered["order_month"] = delivered.order_purchase_timestamp.dt.to_period("M")
monthly = delivered.groupby("order_month").agg(
    orders=("order_id", "nunique"),
    customers=("customer_unique_id", "nunique"),
)
print(monthly, "\n")

orders_per_customer = delivered.groupby("customer_unique_id").order_id.nunique()
print(orders_per_customer.value_counts().sort_index(), "\n")
print("Repeat buyers (%):", round((orders_per_customer > 1).mean() * 100, 2))

Delivered orders: 96478 | missing unique id: 0
             orders  customers
order_month                   
2016-09           1          1
2016-10         265        262
2016-12           1          1
2017-01         750        718
2017-02        1653       1630
2017-03        2546       2508
2017-04        2303       2274
2017-05        3546       3479
2017-06        3135       3076
2017-07        3872       3802
2017-08        4193       4114
2017-09        4150       4083
2017-10        4478       4417
2017-11        7289       7183
2017-12        5513       5450
2018-01        7069       6974
2018-02        6555       6400
2018-03        7003       6914
2018-04        6798       6744
2018-05        6749       6693
2018-06        6099       6061
2018-07        6159       6100
2018-08        6351       6310 

order_id
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64 

Repeat buyers (%): 3.0


### Findings from the monthly volume check
- 3 months have fewer than 300 delivered orders (Sep, Oct, Dec 2016), and Nov 2016 has none. These are dropped.
- Delivered data ends in Aug 2018, so the **analysis window is Jan 2017 to Aug 2018**.
- **Only 3.0% of customers (2,801 of 93,358) ever placed a second delivered order.** Retention will be low in every cohort.
- A handful of customers placed 9 to 15 orders, which suggests business or reseller accounts.
- Nov 2017 is the largest cohort (Black Friday effect).
- Later cohorts have fewer observable months because the data ends in Aug 2018.

## 4. Star Schema: Dimension Tables
**Grain and keys**
- `dim_customers`: one row per real customer (`customer_unique_id`), with a generated surrogate `customer_key`. Location is taken from the customer's most recent order. `cohort_month` is the first delivered-order month over all time (empty if the customer never had a delivered order).
- `dim_date`: one row per calendar day, keyed as `YYYYMMDD`.

In [4]:
# --- dim_customers ---
orders_all = orders.merge(customers, on="customer_id", how="left")

latest_loc = (
    orders_all.sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id").tail(1)
    [["customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"]]
    .rename(columns={"customer_city": "city", "customer_state": "state",
                     "customer_zip_code_prefix": "zip_prefix"})
)
first_delivered = (delivered.groupby("customer_unique_id").order_month.min()
                   .rename("cohort_month").reset_index())

dim_customers = latest_loc.merge(first_delivered, on="customer_unique_id", how="left")
dim_customers.insert(0, "customer_key", range(1, len(dim_customers) + 1))

# --- dim_date ---
start = orders.order_purchase_timestamp.min().normalize().replace(day=1)
end = orders.order_purchase_timestamp.max().normalize() + pd.offsets.MonthEnd(0)
dim_date = pd.DataFrame({"full_date": pd.date_range(start, end)})
dim_date["date_key"] = dim_date.full_date.dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date.full_date.dt.year
dim_date["month"] = dim_date.full_date.dt.month
dim_date["month_start"] = dim_date.full_date.dt.to_period("M").dt.to_timestamp()
dim_date["day_name"] = dim_date.full_date.dt.day_name()
dim_date["is_weekend"] = dim_date.full_date.dt.dayofweek >= 5
dim_date = dim_date[["date_key", "full_date", "year", "month", "month_start", "day_name", "is_weekend"]]

# --- integrity checks ---
assert dim_customers.customer_unique_id.is_unique and dim_date.date_key.is_unique
print("dim_customers:", dim_customers.shape, "| null cohort:", dim_customers.cohort_month.isna().sum())
print("dim_date:", dim_date.shape, "|", dim_date.full_date.min().date(), "->", dim_date.full_date.max().date())

dim_customers.to_csv(PROCESSED_DIR / "dim_customers.csv", index=False)
dim_date.to_csv(PROCESSED_DIR / "dim_date.csv", index=False)
dim_customers.head()

dim_customers: (96096, 6) | null cohort: 2738
dim_date: (791, 7) | 2016-09-01 -> 2018-10-31


,customer_key,customer_unique_id,city,state,zip_prefix,cohort_month
0,1,b7d76e111c89f7ebf14761390f0f7d17,boa vista,RR,69309,NaT
1,2,4854e9b3feff728c13ee5fc7d1547e92,passo fundo,RS,99025,NaT
2,3,009b0127b727ab0ba422f6d9604487c7,sao jose dos campos,SP,12244,NaT
3,4,830d5b7aaa3b6f1e9ad63703bec97d23,sao joaquim da barra,SP,14600,2016-09
4,5,0eb1ee9dba87f5b36b4613a65074337c,sao paulo,SP,2975,NaT


### Findings from the dimension tables
- `dim_customers`: **96,096 rows**, one per real customer. 2,738 customers have no delivered order and therefore no cohort.
- `dim_date`: **791 days** (Sep 2016 to Oct 2018), keyed `YYYYMMDD`.
- Zip prefixes are stored as zero-padded 5-character strings (leading zeros were lost on load).
- Surrogate keys are assigned in order of each customer's latest order and carry no meaning.

## 5. Star Schema: Fact Tables
- `fact_payments`: one row per payment (`order_id` + `payment_sequential`), with its own `customer_key` and `date_key`.
- `fact_orders`: one row per order, with payments aggregated to order level (total value, number of payment rows, max installments). All order statuses are kept so analyses can filter on `order_status`.
- **Integrity checks:** unique grain, no missing keys, and totals that reconcile to the raw payments table.

In [5]:
import numpy as np

# Fix zip prefix and re-save
dim_customers["zip_prefix"] = dim_customers.zip_prefix.astype(str).str.zfill(5)
dim_customers.to_csv(PROCESSED_DIR / "dim_customers.csv", index=False)

# Demonstrate the grain problem with a naive join
print("Naive orders x payments join:", len(orders.merge(payments, on="order_id")), "rows for", len(orders), "orders\n")

# Keys for every order
key_map = dim_customers.set_index("customer_unique_id").customer_key
orders_keyed = orders_all.assign(
    customer_key=orders_all.customer_unique_id.map(key_map),
    date_key=orders_all.order_purchase_timestamp.dt.strftime("%Y%m%d").astype(int),
)[["order_id", "customer_key", "date_key", "order_status"]]

# fact_payments
fact_payments = (
    payments.merge(orders_keyed[["order_id", "customer_key", "date_key"]], on="order_id", how="left")
    .rename(columns={"payment_installments": "installments"})
    [["order_id", "payment_sequential", "customer_key", "date_key",
      "payment_type", "installments", "payment_value"]]
)

# fact_orders (payments aggregated to one row per order)
pay_agg = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    n_payment_rows=("payment_sequential", "count"),
    max_installments=("payment_installments", "max"),
).reset_index()

fact_orders = (
    orders_keyed.rename(columns={"date_key": "order_date_key"})
    .merge(pay_agg, on="order_id", how="left")
)
agg_cols = ["total_payment_value", "n_payment_rows", "max_installments"]
fact_orders[agg_cols] = fact_orders[agg_cols].fillna(0)
fact_orders[["n_payment_rows", "max_installments"]] = fact_orders[["n_payment_rows", "max_installments"]].astype(int)

# Integrity checks
assert fact_orders.order_id.is_unique
assert not fact_payments.duplicated(["order_id", "payment_sequential"]).any()
assert fact_orders.customer_key.notna().all() and fact_payments.customer_key.notna().all()
assert np.isclose(fact_orders.total_payment_value.sum(), payments.payment_value.sum())
assert np.isclose(fact_payments.payment_value.sum(), payments.payment_value.sum())

print("fact_orders:", fact_orders.shape, "| fact_payments:", fact_payments.shape)
print("Total payment value (BRL):", round(fact_payments.payment_value.sum(), 2))
print("Orders with no payment rows:", (fact_orders.n_payment_rows == 0).sum())
print(fact_payments.payment_type.value_counts())

fact_orders.to_csv(PROCESSED_DIR / "fact_orders.csv", index=False)
fact_payments.to_csv(PROCESSED_DIR / "fact_payments.csv", index=False)

Naive orders x payments join: 103886 rows for 99441 orders

fact_orders: (99441, 7) | fact_payments: (103886, 7)
Total payment value (BRL): 16008872.12
Orders with no payment rows: 1
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


### Findings from the fact tables
- `fact_orders`: **99,441 rows** (one per order). `fact_payments`: **103,886 rows** (one per payment). Both reconcile to the raw payments table (total **BRL 16.0M**).
- A naive orders-to-payments join produces 4,445 extra rows. Aggregating payments to order level first keeps the order grain.
- Total payment value includes freight and non-delivered orders, so revenue analysis filters to `delivered`.
- Credit card is about 74% of payments, boleto 19%, voucher 6%, debit 1.5%.
- One order has no payment rows, and 3 payments have type `not_defined`. Both are noted as source-data quirks.

## 6. Schema Extension: Delivery and Reviews
Bonus analysis (delivery time vs review score) needs data the schema does not have yet. Before modeling, inspect the review table's grain and the extra date columns, and check that `dim_date` covers them.

In [7]:
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
orders_full = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv", parse_dates=date_cols)
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv",
                      parse_dates=["review_creation_date", "review_answer_timestamp"])

print("reviews:", reviews.shape)
print("unique review_id:", reviews.review_id.nunique(), "| unique order_id:", reviews.order_id.nunique())
print("duplicate (review_id, order_id) pairs:", reviews.duplicated(["review_id", "order_id"]).sum())
print("orders with more than one review:", (reviews.groupby("order_id").size() > 1).sum())
print("reviews pointing to unknown orders:", (~reviews.order_id.isin(orders_full.order_id)).sum())
print(reviews.review_score.value_counts().sort_index(), "\n")

dlv = orders_full[orders_full.order_status == "delivered"]
print("Delivered orders missing a delivery date:", dlv.order_delivered_customer_date.isna().sum())
print("Delivered before purchase:", (dlv.order_delivered_customer_date < dlv.order_purchase_timestamp).sum(), "\n")

print("Latest value per date column:")
print(orders_full[date_cols].max(), "\n")
print("dim_date currently ends:", dim_date.full_date.max().date())

reviews: (99224, 7)
unique review_id: 98410 | unique order_id: 98673
duplicate (review_id, order_id) pairs: 0
orders with more than one review: 547
reviews pointing to unknown orders: 0
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64 

Delivered orders missing a delivery date: 8
Delivered before purchase: 0 

Latest value per date column:
order_purchase_timestamp        2018-10-17 17:30:18
order_approved_at               2018-09-03 17:40:06
order_delivered_carrier_date    2018-09-11 19:48:28
order_delivered_customer_date   2018-10-17 13:22:46
order_estimated_delivery_date   2018-11-12 00:00:00
dtype: datetime64[us] 

dim_date currently ends: 2018-10-31


### Findings from the review and delivery inspection
- **99,224 reviews** cover 98,673 orders. 768 orders have no review.
- `review_id` alone is not unique (814 rows share an ID with another row) and 547 orders have more than one review. The natural key is (`review_id`, `order_id`). All reviews point to existing orders.
- Scores are polarized: 57.8% are 5 stars, 11.5% are 1 star, and 2-star reviews are rare. Report the share of negative reviews as well as the average.
- 8 delivered orders have no delivery date. They are excluded from delivery-time averages.
- The latest estimated delivery date (2018-11-12) is after the end of `dim_date` (2018-10-31), so `dim_date` must be extended before delivery date keys can be added.

### 6.1 Extend `dim_date`
Rebuild the date dimension so it covers every date used by the new foreign keys (delivery, estimated delivery and review dates). This replaces the `dim_date` built in section 4, and an assertion confirms that no date in the source data is missing from it.

In [8]:
review_cols = ["review_creation_date", "review_answer_timestamp"]
earliest = min(orders_full[date_cols].min().min(), reviews[review_cols].min().min())
latest   = max(orders_full[date_cols].max().max(), reviews[review_cols].max().max())

start = earliest.normalize().replace(day=1)
end   = latest.normalize() + pd.offsets.MonthEnd(0)

dim_date = pd.DataFrame({"full_date": pd.date_range(start, end)})
dim_date["date_key"] = dim_date.full_date.dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date.full_date.dt.year
dim_date["month"] = dim_date.full_date.dt.month
dim_date["month_start"] = dim_date.full_date.dt.to_period("M").dt.to_timestamp()
dim_date["day_name"] = dim_date.full_date.dt.day_name()
dim_date["is_weekend"] = dim_date.full_date.dt.dayofweek >= 5
dim_date = dim_date[["date_key", "full_date", "year", "month", "month_start", "day_name", "is_weekend"]]

# Checks: unique keys, and every source date has a matching key
assert dim_date.date_key.is_unique
for col in date_cols + review_cols:
    src = orders_full[col] if col in date_cols else reviews[col]
    keys = src.dropna().dt.strftime("%Y%m%d").astype(int)
    assert keys.isin(dim_date.date_key).all(), f"missing dates for {col}"

print("dim_date:", dim_date.shape, "|", dim_date.full_date.min().date(), "->", dim_date.full_date.max().date())
dim_date.to_csv(PROCESSED_DIR / "dim_date.csv", index=False)

dim_date: (821, 7) | 2016-09-01 -> 2018-11-30


### Result: extended `dim_date`
`dim_date` now has **821 days (2016-09-01 to 2018-11-30)**. The assertion confirms that every order and review timestamp in the source data has a matching date key, so the new delivery, estimated-delivery and review foreign keys can be loaded without orphans.

### 6.2 Extend `fact_orders`
Add delivery information to the order fact:
- `delivered_date_key`, `estimated_date_key`: keys into `dim_date` (the delivery key is empty for undelivered orders).
- `delivery_days`: elapsed days from purchase to delivery.
- `delay_days`: calendar days between the estimated and actual delivery date (positive = late, 0 = on the estimated day, negative = early).
Row count and order grain must be unchanged.

In [9]:
base_cols = ["order_id", "customer_key", "order_date_key", "order_status",
             "total_payment_value", "n_payment_rows", "max_installments"]
dates = orders_full[["order_id", "order_purchase_timestamp",
                     "order_delivered_customer_date", "order_estimated_delivery_date"]]

fo = fact_orders[base_cols].merge(dates, on="order_id", how="left")

def to_key(ts):
    return pd.to_numeric(ts.dt.strftime("%Y%m%d"), errors="coerce").astype("Int64")

fo["delivered_date_key"] = to_key(fo.order_delivered_customer_date)
fo["estimated_date_key"] = to_key(fo.order_estimated_delivery_date)
fo["delivery_days"] = ((fo.order_delivered_customer_date - fo.order_purchase_timestamp)
                       .dt.total_seconds() / 86400).round(2)
fo["delay_days"] = (fo.order_delivered_customer_date.dt.normalize()
                    - fo.order_estimated_delivery_date).dt.days

fact_orders = fo[["order_id", "customer_key", "order_date_key", "delivered_date_key",
                  "estimated_date_key", "order_status", "total_payment_value",
                  "n_payment_rows", "max_installments", "delivery_days", "delay_days"]]

# Checks
assert len(fact_orders) == 99441 and fact_orders.order_id.is_unique
assert fact_orders.estimated_date_key.notna().all()
for col in ["delivered_date_key", "estimated_date_key"]:
    assert fact_orders[col].dropna().isin(dim_date.date_key).all()

dl = fact_orders[fact_orders.order_status == "delivered"]
print("fact_orders:", fact_orders.shape)
print("Delivered orders with no delivery time:", dl.delivery_days.isna().sum())
print(dl.delivery_days.describe().round(1), "\n")
print("Late deliveries (% of delivered):", round((dl.delay_days > 0).mean() * 100, 1))

fact_orders.to_csv(PROCESSED_DIR / "fact_orders.csv", index=False)
fact_orders.head()

fact_orders: (99441, 11)
Delivered orders with no delivery time: 8
count    96470.0
mean        12.6
std          9.5
min          0.5
25%          6.8
50%         10.2
75%         15.7
max        209.6
Name: delivery_days, dtype: float64 

Late deliveries (% of delivered): 6.8


,order_id,customer_key,order_date_key,delivered_date_key,estimated_date_key,order_status,total_payment_value,n_payment_rows,max_installments,delivery_days,delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,26288,20171002,20171010,20171018,delivered,38.71,3,1,8.44,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,87940,20180724,20180807,20180813,delivered,141.46,1,1,13.78,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,91820,20180808,20180817,20180904,delivered,179.12,1,3,9.39,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,33425,20171118,20171202,20171215,delivered,72.20,1,1,13.21,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,53096,20180213,20180216,20180226,delivered,28.62,1,1,2.87,-10.0


### 6.3 Create `fact_reviews`
Grain: one row per (`review_id`, `order_id`), so all source rows are kept. Adds `customer_key` (via the order) and `review_date_key` (review creation date). `review_answered_at` is kept so the analysis can choose each order's latest review. Free-text comments are not needed for this analysis and are left out.

In [12]:
fr = reviews.merge(fact_orders[["order_id", "customer_key"]], on="order_id", how="left")
fr["review_date_key"] = fr.review_creation_date.dt.strftime("%Y%m%d").astype(int)

fact_reviews = (fr.rename(columns={"review_answer_timestamp": "review_answered_at"})
                [["review_id", "order_id", "customer_key", "review_date_key",
                  "review_score", "review_answered_at"]])

assert len(fact_reviews) == 99224
assert not fact_reviews.duplicated(["review_id", "order_id"]).any()
assert fact_reviews.customer_key.notna().all()
assert fact_reviews.review_date_key.isin(dim_date.date_key).all()
assert fact_reviews.review_score.between(1, 5).all()

print("fact_reviews:", fact_reviews.shape)
print("Orders reviewed:", fact_reviews.order_id.nunique())
print("Average score (all rows):", round(fact_reviews.review_score.mean(), 2))

fact_reviews.to_csv(PROCESSED_DIR / "fact_reviews.csv", index=False)
fact_reviews.head()

fact_reviews: (99224, 6)
Orders reviewed: 98673
Average score (all rows): 4.09


,review_id,order_id,customer_key,review_date_key,review_score,review_answered_at
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,45612,20180118,4,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,56609,20180310,5,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,50825,20180217,5,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5953,20170421,5,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,52405,20180301,5,2018-03-02 10:26:53


### Result: `fact_reviews`
- **99,224 rows**, one per (`review_id`, `order_id`), covering **98,673 orders**. All source rows are kept, and every review has a customer key and a valid date key.
- `review_answered_at` is kept so the analysis can select each order's latest review (one score per order) without altering the table.
- The raw average score across rows is 4.09. It hides a polarized distribution (57.8% 5-star, 11.5% 1-star), so the analysis reports medians and the share of negative reviews alongside it.

## 7. Load the Star Schema into SQLite
- Schema defined in `sql/00_schema.sql` with primary keys, foreign keys and `NOT NULL` constraints.
- Foreign keys are enforced on load, so any orphan row would raise an error.
- First join query (`sql/01_monthly_summary.sql`) must reproduce the monthly delivered orders and customers from section 3.

In [14]:
import sqlite3

schema_sql = """
DROP VIEW  IF EXISTS v_order_review;
DROP TABLE IF EXISTS fact_reviews;
DROP TABLE IF EXISTS fact_payments;
DROP TABLE IF EXISTS fact_orders;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_customers;

CREATE TABLE dim_customers (
    customer_key       INTEGER PRIMARY KEY,
    customer_unique_id TEXT NOT NULL UNIQUE,
    city TEXT, state TEXT, zip_prefix TEXT, cohort_month TEXT
);
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL,
    year INTEGER, month INTEGER, month_start TEXT, day_name TEXT, is_weekend INTEGER
);
CREATE TABLE fact_orders (
    order_id TEXT PRIMARY KEY,
    customer_key       INTEGER NOT NULL REFERENCES dim_customers(customer_key),
    order_date_key     INTEGER NOT NULL REFERENCES dim_date(date_key),
    delivered_date_key INTEGER REFERENCES dim_date(date_key),
    estimated_date_key INTEGER NOT NULL REFERENCES dim_date(date_key),
    order_status TEXT, total_payment_value REAL, n_payment_rows INTEGER, max_installments INTEGER,
    delivery_days REAL, delay_days INTEGER
);
CREATE TABLE fact_payments (
    order_id TEXT NOT NULL REFERENCES fact_orders(order_id),
    payment_sequential INTEGER NOT NULL,
    customer_key INTEGER NOT NULL REFERENCES dim_customers(customer_key),
    date_key     INTEGER NOT NULL REFERENCES dim_date(date_key),
    payment_type TEXT, installments INTEGER, payment_value REAL,
    PRIMARY KEY (order_id, payment_sequential)
);
CREATE TABLE fact_reviews (
    review_id TEXT NOT NULL,
    order_id  TEXT NOT NULL REFERENCES fact_orders(order_id),
    customer_key    INTEGER NOT NULL REFERENCES dim_customers(customer_key),
    review_date_key INTEGER NOT NULL REFERENCES dim_date(date_key),
    review_score    INTEGER NOT NULL CHECK (review_score BETWEEN 1 AND 5),
    review_answered_at TEXT,
    PRIMARY KEY (review_id, order_id)
);
CREATE VIEW v_order_review AS
SELECT order_id, review_score
FROM (
    SELECT order_id, review_score,
           ROW_NUMBER() OVER (PARTITION BY order_id
                              ORDER BY review_answered_at DESC, review_id) AS rn
    FROM fact_reviews
)
WHERE rn = 1;
"""
(PROJECT_ROOT / "sql" / "00_schema.sql").write_text(schema_sql)

DB_PATH = PROCESSED_DIR / "olist_star.db"
dc = dim_customers.assign(cohort_month=dim_customers.cohort_month.dt.strftime("%Y-%m"))

with sqlite3.connect(DB_PATH) as con:
    con.execute("PRAGMA foreign_keys = ON")
    con.executescript(schema_sql)
    for name, df in [("dim_customers", dc), ("dim_date", dim_date), ("fact_orders", fact_orders),
                     ("fact_payments", fact_payments), ("fact_reviews", fact_reviews)]:
        df.to_sql(name, con, if_exists="append", index=False)
    print({t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
           for t in ["dim_customers", "dim_date", "fact_orders", "fact_payments",
                     "fact_reviews", "v_order_review"]})
    print("Foreign key violations:", len(con.execute("PRAGMA foreign_key_check").fetchall()))

query_sql = """
SELECT d.year, d.month,
       COUNT(DISTINCT o.order_id)           AS orders,
       COUNT(DISTINCT c.customer_unique_id) AS customers,
       ROUND(SUM(o.total_payment_value), 2) AS payment_value
FROM fact_orders o
JOIN dim_customers c ON o.customer_key = c.customer_key
JOIN dim_date d      ON o.order_date_key = d.date_key
WHERE o.order_status = 'delivered'
GROUP BY d.year, d.month
ORDER BY d.year, d.month;
"""
(PROJECT_ROOT / "sql" / "01_monthly_summary.sql").write_text(query_sql)

with sqlite3.connect(DB_PATH) as con:
    monthly_sql = pd.read_sql(query_sql, con)
print(monthly_sql.shape)

{'dim_customers': 96096, 'dim_date': 821, 'fact_orders': 99441, 'fact_payments': 103886, 'fact_reviews': 99224, 'v_order_review': 98673}
Foreign key violations: 0
(23, 5)


### Findings from the SQLite load
- Final star schema in SQLite: **2 dimensions** (`dim_customers`: 96,096; `dim_date`: 821), **3 facts** (`fact_orders`: 99,441; `fact_payments`: 103,886; `fact_reviews`: 99,224) and **1 view** (`v_order_review`: 98,673). Primary keys, foreign keys and a score range check are defined in `sql/00_schema.sql`.
- Foreign keys were enforced during the load, and `PRAGMA foreign_key_check` found **0 violations**.
- `v_order_review` implements the one-score-per-order rule: each order's most recent review (`review_answered_at`, ties broken by `review_id`).
- The monthly star-join query still reproduces the 23 monthly order and customer counts from the raw-data analysis.
- Delivered orders total about **BRL 15.4M**, with the peak month in Nov 2017 (BRL 1.15M). Monthly orders plateau at 6,000 to 7,000 through 2018.
- Source-data quirks: the only order with no payment rows is the sole delivered order of Sep 2016 (outside the analysis window), and the 3 `not_defined` payments all have value 0.